In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
import numpy as np
import datetime as dt
from src.imputation import repeated_imputation
from src.missingness import repeated_simulated_missingness
from src.evaluation import get_errors, measure_reconstruction

In [3]:
df = pd.read_csv(PROJECT_ROOT / "data" / "ETTm1.csv", parse_dates=["date"], index_col="date")

# Imputation Pipeline

Use only train data.

### Normalisation

Necessary as imputed values are from different features, so aggregation of reconstruction accuracy between features requires a commonly scaling.

This allows for the use of MAE instead of MAPE.

In [4]:
from sklearn.preprocessing import MinMaxScaler

scaleMinMax = MinMaxScaler(feature_range=(0, 1))

train = scaleMinMax.fit_transform(df.iloc[:1000])

train = pd.DataFrame(train, columns=df.columns).set_index(df.iloc[:1000].index)


### Simulate Missingness with Masks

In [5]:
mechanisms = ["mar", "mcar"]
proportions = [10, 30, 50]

missing_dict, mask_dict = repeated_simulated_missingness(df=train, mechanisms=mechanisms, proportions=proportions, iterations=10)

In [8]:
missing_dict["mar50"][1].shape

(1000, 7)

In [10]:
missing_dict["mar50"][4]

,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
date,,,,,,,
2016-07-01 00:00:00,0.123774,0.037746,0.039879,0.035678,0.499877,0.081808,0.622304
2016-07-01 00:15:00,0.119010,0.047183,0.030643,0.026633,0.514825,0.114639,0.619180
2016-07-01 00:30:00,0.119010,0.028310,0.030643,0.017839,0.507474,0.065662,0.600607
2016-07-01 00:45:00,0.119010,0.028310,0.030643,0.026633,0.507474,0.065662,0.467476
2016-07-01 01:00:00,0.114247,0.047183,0.030643,0.026633,0.484930,0.098493,0.501540
...,...,...,...,...,...,...,...
2016-07-11 08:45:00,0.328523,0.141549,0.401813,0.223116,0.208772,0.114639,0.674897
2016-07-11 09:00:00,0.428551,0.301972,NaN,0.339196,0.149228,0.098493,NaN
2016-07-11 09:15:00,0.390445,0.235915,NaN,0.250000,NaN,0.278794,0.653244


### Imputed With Models

In [16]:
models = ["locf", "ks", "arima", "lstm"]

imputed_data = repeated_imputation(models=models, missing_dict=missing_dict)

### Calculate Errors in Reconstructed Values

In [ ]:
errors = get_errors(df=train, imputed_data=imputed_data, mask_dict=mask_dict, models=models)

### Derive Reconstruction Metrics

In [ ]:
reconstruction = measure_reconstruction(errors=errors, models=models)